# 00 — Google Colab Setup, `ProjectConfig` Validation & Workflow Navigator

Start here when onboarding to **`LC_Framework`** in Google Colab or a local Jupyter environment.

This notebook walks through:
1. **Modular installation** — install only the slice your task needs (`core`, `[gee]`, `[ml]`, `[obia]`, `[viz]`, or external `luma-stack`).
2. **Optional Google Drive & Earth Engine authentication** (only needed for GEE notebooks `01`, `02`, `04` Section 3A, and `05` Section 3A).
3. **Creating or loading a validated `ProjectConfig` + `Class Schema`** (`configs/example_project.yaml` or inline builder serialized to YAML).
4. **Verifying environment health & optional extras**, followed by a complete stage-notebook index.

In [ ]:
# 1. Modular Installation (uncomment ONLY the slice you need in Google Colab)
# Replace <your-org> with your company GitHub organization.

# Option A — Lean Core only (Hybrid Classification Assembly, Evaluation, Config & I/O — zero GEE/ML deps):
# !pip install -q "git+https://github.com/<your-org>/LC_Framework.git"

# Option B — GEE Workflows (CCDC Temporal Segments, Multi-Sensor Features, AlphaEarth, GEE RF, GEE SNIC):
# !pip install -q "lc-framework[gee] @ git+https://github.com/<your-org>/LC_Framework.git"

# Option C — Local Python ML & RFECV + Local GEOBIA:
# !pip install -q "lc-framework[ml,obia,viz] @ git+https://github.com/<your-org>/LC_Framework.git"

# Option D — Everything including external luma-stack:
# !pip install -q "lc-framework[gee,ml,obia,viz] @ git+https://github.com/<your-org>/LC_Framework.git" "git+https://github.com/<your-org>/luma-stack.git"

from pathlib import Path
import importlib
import tempfile
import lc_framework
from lc_framework.config import ClassDefinition, ClassSchema, EpochConfig, ProjectConfig

print(f"LC_Framework version: {lc_framework.__version__}")

## 2. Optional Google Drive Mount & GEE Authentication
Leave `MOUNT_DRIVE = False` and `AUTHENTICATE_GEE = False` if you are running offline, evaluating pre-exported COGs, or testing configuration validation locally.

In [ ]:
MOUNT_DRIVE = False
AUTHENTICATE_GEE = False
GEE_PROJECT_ID = "your-gcp-project-id"

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

if AUTHENTICATE_GEE:
    import ee
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT_ID)
    print("Earth Engine initialized:", GEE_PROJECT_ID)
else:
    print("Skipping GEE auth (AUTHENTICATE_GEE=False).")

## 3. Load or Build & Validate a Study-Area `ProjectConfig`
All study-area paths, CRS, resolution, epochs, and **`Class Schema`** definitions live in a validated YAML/JSON config — never hardcoded inside library code. Below we load `configs/example_project.yaml` when present, or build and serialize a fresh `ProjectConfig` cleanly.

In [ ]:
EXAMPLE_YAML = Path("../configs/example_project.yaml")
if EXAMPLE_YAML.exists():
    cfg = ProjectConfig.from_yaml(EXAMPLE_YAML)
    print("Loaded ProjectConfig from", EXAMPLE_YAML)
else:
    cfg = ProjectConfig(
        project_name="starter_lulc_project",
        crs="EPSG:32748",
        scale_m=10.0,
        aoi_asset="projects/your-project/assets/aoi",
        export_prefix="projects/your-project/assets/lulc_outputs",
        class_schema=ClassSchema(
            classes=(
                ClassDefinition(id=1, name="Forest", color="#1b5e20"),
                ClassDefinition(id=2, name="Agroforestry", color="#66bb6a"),
                ClassDefinition(id=3, name="Cropland", color="#fbc02d"),
                ClassDefinition(id=4, name="Built-up", color="#e53935"),
                ClassDefinition(id=5, name="Waterbody", color="#0288d1"),
            )
        ),
        epochs=(
            EpochConfig(label="2020", start_date="2020-01-01", end_date="2020-12-31"),
            EpochConfig(label="2025", start_date="2025-01-01", end_date="2025-12-31"),
        ),
        smoothness_weights={"Forest": 3.5, "Agroforestry": 1.5, "Cropland": 1.0, "Built-up": 1.0, "Waterbody": 4.0},
    )
    tmp_yaml = Path(tempfile.mkdtemp(prefix="lc_cfg_")) / "project.yaml"
    cfg.to_yaml(tmp_yaml)
    cfg = ProjectConfig.from_yaml(tmp_yaml)
    print("Created & validated standalone ProjectConfig ->", tmp_yaml)

print("Project:", cfg.project_name, "| CRS:", cfg.crs, "| Scale:", cfg.scale_m, "m")
print("GEE prob bands:", cfg.class_schema.gee_prob_band_names)
print("COG band descriptions:", cfg.class_schema.geotiff_band_descriptions)

## 4. Environment & Optional Extra Diagnostic Check
Inspects which stage extras are currently available in this runtime (missing optional packages simply show as `[not installed]` unless your chosen notebook needs them).

In [ ]:
OPTIONAL_MODULES = {
    "[gee] (earthengine-api)": "ee",
    "[ml] (scikit-learn)": "sklearn",
    "[ml] (lightgbm)": "lightgbm",
    "[ml] (xgboost)": "xgboost",
    "[obia] (scikit-image)": "skimage",
    "[ml/obia] (geopandas)": "geopandas",
    "[viz] (plotly)": "plotly",
    "external (luma-stack)": "luma_ge",
}
for label, mod_name in OPTIONAL_MODULES.items():
    try:
        mod = importlib.import_module(mod_name)
        ver = getattr(mod, "__version__", "installed")
        print(f"  [OK] {label:<26} -> {ver}")
    except ImportError:
        print(f"  [--] {label:<26} -> not installed")

## 5. Stage Notebook Index

| Notebook | Stage | Extra Required |
| :--- | :--- | :--- |
| `00_colab_setup_and_config.ipynb` | Colab bootstrap & `ProjectConfig` validation | Core |
| `01_gee_ccdc_and_features.ipynb` | CCDC `Temporal Segment` synthesis & Multi-Sensor Feature Stack + CSV | `[gee]` |
| `02_sample_augmentation_alphaearth.ipynb` | GEE AlphaEarth embedding `Sample Augmentation` | `[gee]` |
| `03_feature_selection_rfecv.ipynb` | Local CSV-based RFECV feature selection | `[ml]` |
| `04_pixel_classification_gee_and_local.ipynb` | GEE `MULTIPROBABILITY` RF & Local ML backends | `[gee]` and/or `[ml]` |
| `05_segmentation_and_geobia.ipynb` | GEE SNIC / Local SLIC segmentation & Pure `Geo-Object Classification` | `[gee]` / `[obia]` |
| `06_hybrid_classification_assembly.ipynb` | Standalone `Object-Refined Pixel Probabilities` (`Classification Assembly`) | **Core only** |
| `07_accuracy_and_trajectory_evaluation.ipynb` | Olofsson area accuracy, log-loss, active learning & Sankey | **Core only** (optional `[viz]`) |

For EECU optimization and troubleshooting tips, see [`docs/api/colab_guide.md`](../docs/api/colab_guide.md) and [`docs/methodology/01_paradigm_selection_guide.md`](../docs/methodology/01_paradigm_selection_guide.md).